In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.optim as optim

In [25]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print(train_df.shape)
print(test_df.shape)

(891, 12)
(418, 11)


In [26]:
print(train_df.isnull().sum())
print(test_df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


In [27]:
train_df["Age"] = train_df["Age"].fillna(
    train_df["Age"].median()
)

test_df["Age"] = test_df["Age"].fillna(
    test_df["Age"].median()
)

train_df["Embarked"] = train_df["Embarked"].fillna(
    train_df["Embarked"].mode()[0]
)

test_df["Fare"] = test_df["Fare"].fillna(
    test_df["Fare"].median()
)

In [28]:
sex_encoder = LabelEncoder()
embarked_encoder = LabelEncoder()

In [29]:
train_df["Sex"] = sex_encoder.fit_transform(
    train_df["Sex"]
)

test_df["Sex"] = sex_encoder.transform(
    test_df["Sex"]
)

In [30]:
train_df["Embarked"] = embarked_encoder.fit_transform(
    train_df["Embarked"]
)

test_df["Embarked"] = embarked_encoder.transform(
    test_df["Embarked"]
)

In [31]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

In [32]:
X = train_df[features]

y = train_df["Survived"]

X_test_final = test_df[features]


In [33]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

X_test_final = scaler.transform(
    X_test_final
)

In [34]:
print(np.isnan(X).sum())
print(np.isnan(X_test_final).sum())

0
0


In [35]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [36]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_val = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
).reshape(-1,1)

y_val = torch.tensor(
    y_val.values,
    dtype=torch.float32
).reshape(-1,1)

In [37]:
print(torch.isnan(X_train).sum())
print(torch.isnan(y_train).sum())

tensor(0)
tensor(0)


In [38]:
class LogisticRegressionModel(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.linear = nn.Linear(
            input_size,
            1
        )

    def forward(self, x):
        return self.linear(x)

In [39]:
linear_model = LogisticRegressionModel(
    X_train.shape[1]
)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    linear_model.parameters(),
    lr=0.01
)

In [40]:
epochs = 500

for epoch in range(epochs):

    outputs = linear_model(X_train)

    loss = criterion(
        outputs,
        y_train
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch}, Loss={loss.item():.4f}"
        )

Epoch 0, Loss=0.7306
Epoch 50, Loss=0.5211
Epoch 100, Loss=0.4756
Epoch 150, Loss=0.4581
Epoch 200, Loss=0.4515
Epoch 250, Loss=0.4492
Epoch 300, Loss=0.4485
Epoch 350, Loss=0.4483
Epoch 400, Loss=0.4482
Epoch 450, Loss=0.4482


In [41]:
with torch.no_grad():

    logits = linear_model(X_val)

    probs = torch.sigmoid(logits)

    preds = (probs >= 0.5).float()

In [42]:
y_true = y_val.numpy()
y_pred = preds.numpy()

lr_accuracy = accuracy_score(y_true,y_pred)
lr_precision = precision_score(y_true,y_pred)
lr_recall = recall_score(y_true,y_pred)
lr_f1 = f1_score(y_true,y_pred)

print("Accuracy :", lr_accuracy)
print("Precision:", lr_precision)
print("Recall   :", lr_recall)
print("F1 Score :", lr_f1)

Accuracy : 0.8100558659217877
Precision: 0.7857142857142857
Recall   : 0.7432432432432432
F1 Score : 0.7638888888888888


In [43]:
class TitanicMLP(nn.Module):

    def __init__(self,input_size):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size,32),
            nn.ReLU(),

            nn.Linear(32,16),
            nn.ReLU(),

            nn.Linear(16,1)

        )

    def forward(self,x):
        return self.network(x)

In [44]:
mlp_model = TitanicMLP(
    X_train.shape[1]
)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=0.001
)

In [45]:
epochs = 500

for epoch in range(epochs):

    outputs = mlp_model(X_train)

    loss = criterion(
        outputs,
        y_train
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch}, Loss={loss.item():.4f}"
        )

Epoch 0, Loss=0.6788
Epoch 50, Loss=0.5696
Epoch 100, Loss=0.4441
Epoch 150, Loss=0.4055
Epoch 200, Loss=0.3871
Epoch 250, Loss=0.3768
Epoch 300, Loss=0.3694
Epoch 350, Loss=0.3632
Epoch 400, Loss=0.3564
Epoch 450, Loss=0.3490


In [46]:
with torch.no_grad():

    logits = mlp_model(X_val)

    probs = torch.sigmoid(logits)

    preds = (probs >= 0.5).float()

In [47]:
y_true = y_val.numpy()
y_pred = preds.numpy()

mlp_accuracy = accuracy_score(y_true,y_pred)
mlp_precision = precision_score(y_true,y_pred)
mlp_recall = recall_score(y_true,y_pred)
mlp_f1 = f1_score(y_true,y_pred)

print("Accuracy :", mlp_accuracy)
print("Precision:", mlp_precision)
print("Recall   :", mlp_recall)
print("F1 Score :", mlp_f1)

Accuracy : 0.8156424581005587
Precision: 0.8253968253968254
Recall   : 0.7027027027027027
F1 Score : 0.7591240875912408


In [48]:
comparison = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "MLP"
    ],

    "Accuracy":[
        lr_accuracy,
        mlp_accuracy
    ],

    "Precision":[
        lr_precision,
        mlp_precision
    ],

    "Recall":[
        lr_recall,
        mlp_recall
    ],

    "F1 Score":[
        lr_f1,
        mlp_f1
    ]

})

print(comparison)

                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.810056   0.785714  0.743243  0.763889
1                  MLP  0.815642   0.825397  0.702703  0.759124


In [49]:
X_test_tensor = torch.tensor(
    X_test_final,
    dtype=torch.float32
)

In [50]:
with torch.no_grad():

    logits = mlp_model(
        X_test_tensor
    )

    probs = torch.sigmoid(
        logits
    )

    predictions = (
        probs >= 0.5
    ).int()

In [51]:
submission = pd.DataFrame({

    "PassengerId":
    test_df["PassengerId"],

    "Survived":
    predictions.numpy().flatten()

})

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully")

submission.csv created successfully
